# Run SCEPTER Scenarios

This notebook is the execution step after `04_test_scepter_inputs.ipynb`. It reads the staged SCEPTER run table with HWSD2 soil and CHIRPS rainfall inputs, writes one run folder/config file per model-unit/scenario combination, and optionally calls an external SCEPTER command.

This notebook is configured to execute SCEPTER after staging real-data configs. Confirm `SCEPTER_COMMAND_TEMPLATE` points to the working SCEPTER command available in the current runtime.


## Workflow

1. **Load staged run table** from `data/scepter_runs/inputs/scepter_runs.csv`.
2. **Select runs** for staging or execution. By default this uses the full staged table from notebook `04`.
3. **Write run configs** under `data/scepter_runs/runs/{run_id}/scepter_config.json`.
4. **Create output folders** under `data/scepter_runs/outputs/{run_id}/`.
5. **Optionally execute SCEPTER** using a command template that receives `{config_path}`, `{run_dir}`, `{output_dir}`, and `{run_id}`.
6. **Write an execution manifest** to `data/scepter_runs/outputs/scepter_execution_manifest.csv`.

The execution manifest is the handoff to the next notebook, which will parse completed SCEPTER outputs and join them back to parcel/model-unit geometries.


In [ ]:
from pathlib import Path
import os
import shutil
import sys

import pandas as pd

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

from erw_mrv.paths import SCEPTER_INPUTS, SCEPTER_OUTPUTS, SCEPTER_RUN_DIR, ensure_dir
from erw_mrv.scepter import (
    execute_scepter_runs,
    load_scepter_runs,
    select_scepter_runs,
    write_run_configs,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)


## Settings

Use `SCEPTER_RUN_MODE` to choose how much work notebook `05` should execute:

- `"smoke"`: one very short baseline run, used only to prove the binary and adapter work.
- `"pilot"`: four runs for the first model unit, baseline plus all ERW scenarios, so notebook `06` can compute additionality before the expensive final batch.
- `"final"`: all staged model-unit/scenario runs, using production years through the adapter.

SCEPTER is expected to be built locally, outside this notebook. Set the `SCEPTER_EXECUTABLE` environment variable, or place the built binary at `external/SCEPTER/scepter` under the project root.


In [ ]:
# Rebuild setup if this cell is run after a kernel restart without the setup cell.
if "SCEPTER_INPUTS" not in globals():
    from pathlib import Path
    import os
    import shutil
    import sys

    import pandas as pd

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ModuleNotFoundError:
        pass

    colab_project_root = Path("/content/drive/MyDrive/erw_spatial_mrv")
    cwd = Path.cwd().resolve()
    candidates = [colab_project_root, cwd, cwd.parent, cwd / "erw_spatial_mrv", cwd.parent / "erw_spatial_mrv"]
    source_project_root = next(
        (candidate for candidate in candidates if (candidate / "src" / "erw_mrv" / "__init__.py").exists()),
        None,
    )
    if source_project_root is None:
        checked = chr(10).join(f"- {candidate}" for candidate in candidates)
        raise ModuleNotFoundError(
            "Could not rebuild notebook setup because src/erw_mrv was not found. "
            "Run the first setup/import cell or sync the full project to /content/drive/MyDrive/erw_spatial_mrv. "
            f"Checked:{chr(10)}{checked}"
        )

    PROJECT_ROOT = colab_project_root if (colab_project_root / "data").exists() else source_project_root
    data_root = PROJECT_ROOT / "data"
    os.environ["ERW_MRV_DATA_ROOT"] = str(data_root)
    src = source_project_root / "src"
    if str(src) not in sys.path:
        sys.path.insert(0, str(src))

    from erw_mrv.paths import SCEPTER_INPUTS, SCEPTER_OUTPUTS, SCEPTER_RUN_DIR, ensure_dir
    from erw_mrv.scepter import execute_scepter_runs, load_scepter_runs, select_scepter_runs, write_run_configs

RUN_TABLE_PATH = SCEPTER_INPUTS / "scepter_runs.csv"
MODEL_UNITS_TABLE_PATH = SCEPTER_INPUTS / "scepter_model_units.csv"
RUN_ROOT = ensure_dir(SCEPTER_RUN_DIR)
OUTPUT_ROOT = ensure_dir(SCEPTER_OUTPUTS)

SCEPTER_RUN_MODE = os.environ.get("ERW_SCEPTER_RUN_MODE", "pilot").lower()
VALID_SCEPTER_RUN_MODES = {"smoke", "pilot", "final"}
if SCEPTER_RUN_MODE not in VALID_SCEPTER_RUN_MODES:
    raise ValueError(f"SCEPTER_RUN_MODE must be one of {sorted(VALID_SCEPTER_RUN_MODES)}, got {SCEPTER_RUN_MODE!r}")

if SCEPTER_RUN_MODE == "smoke":
    MAX_TEST_RUNS = 1
    SCENARIO_IDS = None
    SCEPTER_ADAPTER_EXTRA_ARGS = ""
elif SCEPTER_RUN_MODE == "pilot":
    MAX_TEST_RUNS = 4
    SCENARIO_IDS = None
    SCEPTER_ADAPTER_EXTRA_ARGS = ""
else:
    MAX_TEST_RUNS = None
    SCENARIO_IDS = None
    production_years = os.environ.get("ERW_SCEPTER_PRODUCTION_YEARS", "1")
    SCEPTER_ADAPTER_EXTRA_ARGS = f"--production-years {production_years}"

REQUIRE_REAL_STAGED_INPUTS = True
RUN_EXTERNAL_SCEPTER = True
SCEPTER_SOURCE_ROOT = PROJECT_ROOT / "external" / "SCEPTER"
SCEPTER_EXECUTABLE = Path(os.environ.get("SCEPTER_EXECUTABLE", SCEPTER_SOURCE_ROOT / "scepter")).expanduser()
TIMEOUT_SECONDS = int(os.environ.get("ERW_SCEPTER_TIMEOUT_SECONDS", str(6 * 60 * 60)))

SCEPTER_ADAPTER = PROJECT_ROOT / "scripts" / "run_scepter_adapter.py"

# Build SCEPTER locally outside this notebook, then let the adapter translate JSON configs
# into SCEPTER's native input files before executing the compiled binary.
SCEPTER_COMMAND_TEMPLATE = (
    f"{sys.executable} {SCEPTER_ADAPTER} "
    f"--scepter-root {SCEPTER_SOURCE_ROOT} "
    f"--executable {SCEPTER_EXECUTABLE} "
    "--config {config_path} --output-dir {output_dir} "
    f"--timeout-seconds {TIMEOUT_SECONDS} "
    f"{SCEPTER_ADAPTER_EXTRA_ARGS}"
).strip()

MANIFEST_PATH = OUTPUT_ROOT / "scepter_execution_manifest.csv"
print(f"SCEPTER_RUN_MODE = {SCEPTER_RUN_MODE}")
print(f"MAX_TEST_RUNS = {MAX_TEST_RUNS}")
print(f"SCENARIO_IDS = {SCENARIO_IDS}")
print(f"TIMEOUT_SECONDS = {TIMEOUT_SECONDS}")
print(f"SCEPTER_ADAPTER_EXTRA_ARGS = {SCEPTER_ADAPTER_EXTRA_ARGS or '(smoke-test defaults)'}")
RUN_ROOT, OUTPUT_ROOT, MANIFEST_PATH


## Load Run Table

This table should come from notebook `04`. Each row is one model unit under one ERW scenario.


In [ ]:
runs = load_scepter_runs(RUN_TABLE_PATH)

real_input_columns = ["soil_source", "rainfall_source", "rainfall_months_used", "runoff_note"]


def enrich_runs_with_model_unit_metadata(runs_df: pd.DataFrame) -> pd.DataFrame:
    missing_columns = [column for column in real_input_columns if column not in runs_df.columns]
    if not missing_columns:
        return runs_df

    if not MODEL_UNITS_TABLE_PATH.exists():
        return runs_df

    units = pd.read_csv(MODEL_UNITS_TABLE_PATH)
    available_metadata = [column for column in missing_columns if column in units.columns]
    if not available_metadata:
        return runs_df

    print(
        "Run table is missing real input metadata; enriching from "
        f"{MODEL_UNITS_TABLE_PATH}: {available_metadata}"
    )
    return runs_df.merge(units[["model_unit_id", *available_metadata]], on="model_unit_id", how="left")


runs = enrich_runs_with_model_unit_metadata(runs)
missing_real_input_columns = [column for column in real_input_columns if column not in runs.columns]
if REQUIRE_REAL_STAGED_INPUTS and missing_real_input_columns:
    raise ValueError(
        "The staged SCEPTER run table does not include real input metadata, and it could not be recovered "
        "from scepter_model_units.csv. Rerun notebooks 03a, 03b, and 04 after syncing the updated src/ code. "
        f"Missing columns: {missing_real_input_columns}"
    )

if REQUIRE_REAL_STAGED_INPUTS:
    placeholder_rows = runs.astype(str).apply(lambda col: col.str.contains("placeholder", case=False, na=False)).any(axis=1)
    if placeholder_rows.any():
        raise ValueError(
            f"Found {int(placeholder_rows.sum())} staged runs using placeholder inputs. "
            "Rerun notebook 04 with HWSD2 and CHIRPS artifacts available."
        )

print(f"Available staged runs: {len(runs):,}")
print("Soil source:", runs["soil_source"].dropna().astype(str).iloc[0] if "soil_source" in runs else "not recorded")
print("Rainfall source:", runs["rainfall_source"].dropna().astype(str).iloc[0] if "rainfall_source" in runs else "not recorded")
print("Rainfall months:", runs["rainfall_months_used"].dropna().astype(str).iloc[0] if "rainfall_months_used" in runs else "not recorded")
runs.groupby("scenario_id").size().rename("run_count").reset_index()


## Select Runs For This Batch

`smoke` selects one run, `pilot` selects the first four sorted runs, and `final` selects all staged runs. Run `pilot` before `final` so notebook `06` can verify baseline-vs-scenario additionality for one model unit.


In [ ]:
selected_runs = select_scepter_runs(
    runs,
    max_runs=MAX_TEST_RUNS,
    scenario_ids=SCENARIO_IDS,
)

print(f"Runs selected for this batch: {len(selected_runs):,}")
selected_runs[["run_id", "model_unit_id", "scenario_id", "area_ha", "basalt_application_t_ha", "basalt_d50_um"]].head(20)


## Stage Run Configs

Each run gets a folder under `data/scepter_runs/runs/` and a JSON configuration file. The config includes site, HWSD2 soil inputs, CHIRPS rainfall/runoff inputs, amendment scenario settings, and `input_sources` metadata.


In [ ]:
execution_table = write_run_configs(
    selected_runs,
    run_root=RUN_ROOT,
    output_root=OUTPUT_ROOT,
)

execution_table.head()


## Inspect One Config

Before running an external model, inspect one generated config to confirm units, real-data sources, and scenario parameters look reasonable.


In [ ]:
example_config = Path(execution_table.iloc[0]["config_path"])
print(example_config)
print(example_config.read_text()[:2000])


## Execute Or Stage Manifest

When `RUN_EXTERNAL_SCEPTER = False`, this cell writes a manifest showing that real-data configs were staged but not executed. When set to `True`, it calls the command template once per run.

The command template can use these fields: `{run_id}`, `{run_dir}`, `{output_dir}`, `{config_path}`, `{model_unit_id}`, and `{scenario_id}`.


In [ ]:
if RUN_EXTERNAL_SCEPTER:
    scepter_executable = SCEPTER_COMMAND_TEMPLATE.split()[0]
    if shutil.which(scepter_executable) is None:
        raise RuntimeError(
            f"RUN_EXTERNAL_SCEPTER is True, but the command `{scepter_executable}` is not available in this kernel. "
            "Install SCEPTER in the active Colab/Jupyter runtime, or update SCEPTER_COMMAND_TEMPLATE "
            "to the full executable/script path. If you only want to stage configs for now, set "
            "RUN_EXTERNAL_SCEPTER = False."
        )

    if MANIFEST_PATH.exists():
        previous_manifest = pd.read_csv(MANIFEST_PATH)
        completed_run_ids = set(previous_manifest.loc[previous_manifest["status"] == "complete", "run_id"])
        pending_execution_table = execution_table[~execution_table["run_id"].isin(completed_run_ids)].copy()
        print(f"Resuming SCEPTER execution: {len(completed_run_ids):,} complete, {len(pending_execution_table):,} pending.")
    else:
        previous_manifest = pd.DataFrame()
        completed_run_ids = set()
        pending_execution_table = execution_table.copy()

    if pending_execution_table.empty:
        manifest = previous_manifest.copy()
    else:
        new_manifest = execute_scepter_runs(
            pending_execution_table,
            command_template=SCEPTER_COMMAND_TEMPLATE,
            timeout_seconds=TIMEOUT_SECONDS,
        )
        if previous_manifest.empty:
            manifest = new_manifest
        else:
            previous_keep = previous_manifest[previous_manifest["run_id"].isin(completed_run_ids)].copy()
            manifest = pd.concat([previous_keep, new_manifest], ignore_index=True)
else:
    manifest = execution_table.copy()
    manifest["command"] = SCEPTER_COMMAND_TEMPLATE
    manifest["status"] = "staged_not_executed"
    manifest["return_code"] = pd.NA
    manifest["elapsed_seconds"] = 0.0
    manifest["stdout_log"] = pd.NA
    manifest["stderr_log"] = pd.NA

manifest = manifest.sort_values(["scenario_id", "run_id"]).reset_index(drop=True)
manifest.to_csv(MANIFEST_PATH, index=False)
manifest[["run_id", "scenario_id", "status", "config_path", "output_dir"]].head(20)


## Batch Summary


In [ ]:
status_summary = manifest.groupby("status").size().rename("run_count").reset_index()
scenario_summary = manifest.groupby(["scenario_id", "status"]).size().rename("run_count").reset_index()
source_columns = [column for column in ["soil_source", "rainfall_source", "rainfall_months_used", "missing_requested_months"] if column in manifest.columns]
source_summary = manifest[source_columns].drop_duplicates().reset_index(drop=True) if source_columns else pd.DataFrame()

status_summary, scenario_summary, source_summary, MANIFEST_PATH


## Expected SCEPTER Outputs

Once external execution is enabled, each run folder under `data/scepter_runs/outputs/{run_id}/` should contain whatever files SCEPTER produces for that HWSD2/CHIRPS-backed scenario. The next notebook parses those outputs into a clean table with at least:

- `run_id`
- `model_unit_id`
- `scenario_id`
- weathering or alkalinity metric
- cumulative CO2 removal
- CO2 removal per hectare
- soil pH or chemistry changes if available

The parser will use `scepter_execution_manifest.csv` to know which runs completed and where their output folders are.


## Outputs From This Notebook

This notebook writes SCEPTER batch execution artifacts:

- `data/scepter_runs/runs/{run_id}/scepter_config.json`: one model config per selected run.
- `data/scepter_runs/outputs/{run_id}/`: output folder reserved for each selected run.
- `data/scepter_runs/outputs/scepter_execution_manifest.csv`: status table for staged or executed runs.
- `data/scepter_runs/runs/{run_id}/scepter_stdout.log`: stdout log, written only when `RUN_EXTERNAL_SCEPTER = True`.
- `data/scepter_runs/runs/{run_id}/scepter_stderr.log`: stderr log, written only when `RUN_EXTERNAL_SCEPTER = True`.

The next step is notebook `06`, which should read the execution manifest, parse completed SCEPTER output files, and join model results back to spatial model units.
